# MOD-05 · NB-04 — Word Embeddings & Word2Vec for Clinical Text
### Health Analytics with Python · Module 05: NLP for Clinical Text
---
**Learning objectives**
- Understand word embeddings vs sparse TF-IDF representations
- Train clinical Word2Vec with Skip-gram and CBOW
- Explore semantic similarity in the clinical embedding space
- Build document vectors by averaging word embeddings
- Visualise the clinical embedding space with t-SNE
- Compare Skip-gram vs CBOW for rare medical terms

**Estimated time:** 2.5 hours | **Level:** Intermediate → Advanced | **Libraries:** `gensim`, `sklearn`

## 1. Setup

In [ ]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)

try:
    from gensim.models import Word2Vec
    GENSIM_OK = True
    print("gensim available")
except ImportError:
    GENSIM_OK = False
    print("pip install gensim")


## 2. Clinical Corpus for Embedding Training

In [ ]:
CLINICAL_SENTENCES = [
    "heart failure reduced ejection fraction BNP elevated furosemide diuresis daily weights",
    "myocardial infarction troponin elevated EKG ST elevation cath lab PCI stent coronary",
    "atrial fibrillation rate control metoprolol digoxin anticoagulation warfarin stroke prevention",
    "hypertension lisinopril amlodipine blood pressure end organ damage kidney function",
    "cardiac catheterization coronary artery disease stenosis stent drug eluting stent",
    "COPD exacerbation albuterol ipratropium bronchospasm wheezes oxygen supplementation SpO2",
    "pneumonia consolidation fever WBC elevated ceftriaxone azithromycin blood cultures",
    "pulmonary embolism anticoagulation rivaroxaban heparin DVT bilateral lower extremity",
    "respiratory failure mechanical ventilation intubation PEEP FiO2 arterial blood gas",
    "pleural effusion thoracentesis protein LDH transudative exudative Light criteria",
    "diabetic ketoacidosis insulin drip anion gap metabolic acidosis bicarbonate potassium",
    "type 2 diabetes metformin GLP-1 HbA1c glycemic control SGLT2 cardiovascular renal benefit",
    "hypoglycemia glucose dextrose altered mental status glucagon diabetes education",
    "acute kidney injury creatinine elevated BUN nephrotoxins hold contrast ACE inhibitor",
    "hemodialysis end stage renal disease potassium hyperkalemia access fistula bicarbonate",
    "nephrotic syndrome proteinuria edema albumin low diuretics renal biopsy immunosuppression",
    "urinary tract infection pyuria nitrites E coli culture sensitivity trimethoprim nitrofurantoin",
    "septic shock vasopressor norepinephrine broad spectrum antibiotics vancomycin meropenem lactate",
    "bacteremia blood cultures Staph aureus MSSA nafcillin vancomycin echocardiogram endocarditis",
    "neutropenic fever chemotherapy immunosuppressed filgrastim antifungal fluconazole cultures",
    "ischemic stroke MRI DWI diffusion restriction tPA thrombolysis antiplatelet aspirin",
    "seizure antiepileptic levetiracetam valproate EEG epilepsy status epilepticus",
    "pain management opioid morphine acetaminophen ibuprofen NSAID multimodal analgesia",
    "deep vein thrombosis anticoagulation enoxaparin warfarin compression ambulation prophylaxis",
    "post-operative fever wound infection cellulitis erythema warmth drainage cultures antibiotics",
    "malnutrition albumin pre-albumin total parenteral nutrition nasogastric tube dietitian",
]

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    return text.split()

# Augment corpus
augmented = []
for sent in CLINICAL_SENTENCES:
    base = tokenize(sent)
    augmented.append(base)
    for _ in range(20):
        shuffled = base.copy()
        np.random.shuffle(shuffled)
        augmented.append(shuffled[:max(4, len(shuffled)//2 + np.random.randint(1,5))])

print(f"Training corpus: {len(augmented)} sentences")
print(f"Sample: {augmented[0][:10]}")


## 3. Train Clinical Word2Vec

In [ ]:
if GENSIM_OK:
    # Skip-gram (sg=1) — better for rare medical terms
    sg_model = Word2Vec(sentences=augmented, vector_size=100, window=5,
                        min_count=2, workers=4, sg=1, negative=10,
                        epochs=40, seed=42)
    wv = sg_model.wv
    print(f"Vocabulary: {len(wv.key_to_index)} tokens | Dim: {wv.vector_size}")

    print("\nSemantic similarity examples:")
    for word in ["diabetes","heart","infection","antibiotics","kidney","stroke"]:
        if word in wv:
            similar = wv.most_similar(word, topn=4)
            print(f"  {word:15s}: {[f'{w}({s:.2f})' for w,s in similar]}")
else:
    print("gensim not available — install with: pip install gensim")


## 4. Clinical Analogy Tasks

In [ ]:
if GENSIM_OK and len(wv) > 50:
    analogy_tests = [
        ("diabetes",  "insulin",     "infection"),
        ("heart",     "failure",     "kidney"),
        ("acute",     "kidney",      "heart"),
        ("septic",    "shock",       "cardiogenic"),
    ]
    print("Analogy tasks (A - B + C ≈ ?):\n")
    for A, B, C in analogy_tests:
        if all(w in wv for w in [A,B,C]):
            try:
                results = wv.most_similar(positive=[A,C], negative=[B], topn=3)
                print(f"  {A} - {B} + {C} ≈ {[f'{w}({s:.2f})' for w,s in results]}")
            except Exception as e:
                print(f"  {A} - {B} + {C}: {e}")
        else:
            missing = [w for w in [A,B,C] if w not in wv]
            print(f"  Skipped (not in vocab: {missing})")


## 5. Document Vectors by Averaging

In [ ]:
CONDITION_TEMPLATES = {
    "Cardiac"  : ["heart failure BNP furosemide ejection fraction cardiomegaly edema",
                  "myocardial infarction troponin PCI coronary artery stent aspirin",
                  "atrial fibrillation rate control anticoagulation warfarin apixaban"],
    "Pulmonary": ["COPD bronchospasm albuterol steroids oxygen wheezes exacerbation",
                  "pneumonia consolidation fever antibiotics ceftriaxone cultures",
                  "pulmonary embolism anticoagulation heparin rivaroxaban DVT"],
    "Diabetes" : ["diabetes insulin HbA1c metformin glucose ketoacidosis anion gap",
                  "hypoglycemia dextrose glucagon blood sugar altered mental status",
                  "diabetic foot infection wound amputation vascular antibiotics"],
    "Renal"    : ["acute kidney injury creatinine nephrotoxins dialysis hemodialysis",
                  "proteinuria nephrotic syndrome albumin diuretics renal biopsy",
                  "UTI pyuria culture E coli trimethoprim nitrofurantoin"],
    "Sepsis"   : ["septic shock norepinephrine vasopressor lactate cultures broad spectrum",
                  "bacteremia Staph aureus vancomycin blood cultures endocarditis",
                  "neutropenic fever antifungal immunosuppressed chemotherapy"],
}

doc_data = []
for cond, templates in CONDITION_TEMPLATES.items():
    for _ in range(80):
        base = tokenize(np.random.choice(templates))
        noise = [np.random.choice(["patient","treatment","management","hospital"]) for _ in range(2)]
        tokens = base + noise; np.random.shuffle(tokens)
        doc_data.append({"tokens":tokens,"condition":cond})

df_docs = pd.DataFrame(doc_data)

def get_doc_vector(tokens, word_vectors, dim=100):
    vecs = [word_vectors[t] for t in tokens if t in word_vectors]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

if GENSIM_OK:
    X_w2v = np.array([get_doc_vector(row.tokens, wv) for _, row in df_docs.iterrows()])
    y_docs = df_docs["condition"]
    X_tr, X_te, y_tr, y_te = train_test_split(X_w2v, y_docs, test_size=0.2, stratify=y_docs, random_state=42)
    lr = LogisticRegression(C=1, max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    print(f"Word2Vec doc-vec accuracy: {lr.score(X_te, y_te):.4f}")


## 6. t-SNE Embedding Space Visualisation

In [ ]:
if GENSIM_OK and len(wv) > 30:
    MEDICAL_TERMS = {
        "Cardiac"  : ["heart","failure","troponin","coronary","ejection","stent"],
        "Pulmonary": ["pneumonia","bronchospasm","oxygen","consolidation","wheezes"],
        "Diabetes" : ["diabetes","insulin","glucose","ketoacidosis","metformin"],
        "Renal"    : ["kidney","creatinine","dialysis","proteinuria","nephrotoxins"],
        "Sepsis"   : ["septic","lactate","vasopressor","bacteremia","antibiotics"],
    }
    terms, labels = [], []
    for cond, term_list in MEDICAL_TERMS.items():
        for t in term_list:
            if t in wv:
                terms.append(t); labels.append(cond)

    if len(terms) > 10:
        vectors = np.array([wv[t] for t in terms])
        perp = min(5, len(terms)-1)
        coords = TSNE(n_components=2, random_state=42, perplexity=perp).fit_transform(vectors)

        COLORS = {"Cardiac":"#D65F5F","Pulmonary":"#4878CF","Diabetes":"#6ACC65",
                  "Renal":"#B47CC7","Sepsis":"#D4A017"}
        fig, ax = plt.subplots(figsize=(11,8))
        for i,(term,label) in enumerate(zip(terms,labels)):
            color = COLORS[label]
            ax.scatter(coords[i,0],coords[i,1],c=color,s=80,alpha=0.8,zorder=3)
            ax.annotate(term,(coords[i,0]+0.3,coords[i,1]+0.3),fontsize=8,color=color)
        patches = [mpatches.Patch(color=c,label=l) for l,c in COLORS.items()]
        ax.legend(handles=patches,fontsize=9)
        ax.set_title("t-SNE: Clinical Word2Vec Embedding Space")
        ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
        plt.tight_layout()
        plt.savefig("/tmp/mod05/word2vec_tsne.png",bbox_inches="tight"); plt.show()
        print(f"Plotted {len(terms)} clinical terms")


## 7. Skip-gram vs CBOW Comparison

In [ ]:
if GENSIM_OK:
    cbow = Word2Vec(sentences=augmented, vector_size=100, window=5,
                    min_count=2, workers=4, sg=0, epochs=40, seed=42)
    cbow_wv = cbow.wv

    pairs = [("heart","failure"),("diabetes","insulin"),("septic","shock"),
             ("kidney","dialysis"),("antibiotics","infection")]
    print("Cosine similarity: Skip-gram vs CBOW\n")
    print(f"{'Pair':35s} {'Skip-gram':>12s} {'CBOW':>8s}")
    print("-"*58)
    for w1,w2 in pairs:
        if all(w in wv and w in cbow_wv for w in [w1,w2]):
            sg_s   = wv.similarity(w1,w2)
            cbow_s = cbow_wv.similarity(w1,w2)
            print(f"  {w1} — {w2:25s} {sg_s:>12.4f} {cbow_s:>8.4f}")


## 8. Knowledge Check
1. Why does Skip-gram typically work better for rare medical terminology?
2. Window size 5 was used. Would smaller (2) or larger (10) better capture clinical relationships?
3. How would you verify that 'diabetes' and 'insulin' embeddings are closer than 'diabetes' and 'antibiotics'?
4. Averaging word vectors discards word order. Give two clinical examples where order changes meaning.
5. Train Word2Vec with `min_count=1` — does vocabulary grow significantly? What is the tradeoff?

In [ ]:
if GENSIM_OK:
    model_mc1 = Word2Vec(sentences=augmented, vector_size=100, window=5,
                         min_count=1, workers=4, sg=1, epochs=40, seed=42)
    print(f"min_count=2 vocab: {len(wv.key_to_index)}")
    print(f"min_count=1 vocab: {len(model_mc1.wv.key_to_index)}")
    print(f"Growth: +{len(model_mc1.wv.key_to_index)-len(wv.key_to_index)} rare tokens")
    print("Tradeoff: more coverage but noisy embeddings for very rare terms")


---
**Next → NB-05: Clinical BERT Embeddings**